# 02 — Multi-Sheet Load

Mirrors the customer's original pattern: iterate over all sheets and load each one.

Uses `operation="listSheets"` — a new native capability in DBR 17.1+ that replaces the need to hardcode sheet names.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "salt_river_excel")
UC_VOLUME  = os.getenv("UC_VOLUME",  "raw_files")

FILE_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}/salt_river_fields.xlsx"

In [2]:
# Step 1: discover sheet names natively
sheets_df = spark.read.option("operation", "listSheets").excel(FILE_PATH)
sheets_df.show()

+----------+--------------+
|sheetIndex|     sheetName|
+----------+--------------+
|         0|    Attendance|
|         1|   Concessions|
|         2|   Merchandise|
|         3|Season_Summary|
+----------+--------------+



In [ ]:
# Step 2: load each sheet into a temp view — mirrors the customer's original loop
sheet_names = [row.sheetName for row in sheets_df.collect()]

loaded = {}

for sheet in sheet_names:
    df = (
        spark.read
        .option("dataAddress", sheet)
        .option("headerRows", 1)
        .excel(FILE_PATH)
    )
    view_name = sheet.replace(" ", "_").replace("-", "_")
    df.createOrReplaceTempView(view_name)
    loaded[sheet] = df
    print(f"  '{sheet}' → view '{view_name}'  ({df.count()} rows, {len(df.columns)} cols)")

print("\nAll sheets loaded.")

  'Attendance' → view 'Attendance'  (18 rows, 7 cols)
  'Concessions' → view 'Concessions'  (270 rows, 7 cols)
  'Merchandise' → view 'Merchandise'  (342 rows, 9 cols)
  'Season_Summary' → view 'Season_Summary'  (10 rows, 2 cols)

All sheets loaded.


In [4]:
# Quick sanity checks
print("=== Attendance ===")
spark.sql("SELECT * FROM Attendance ORDER BY game_date").show(5, truncate=False)

print("=== Concessions (top items by revenue) ===")
spark.sql("""
    SELECT item_name, category,
           SUM(CAST(units_sold AS LONG)) AS total_units,
           ROUND(SUM(CAST(total_revenue_usd AS DOUBLE)), 2) AS total_revenue
    FROM Concessions
    GROUP BY item_name, category
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

print("=== Merchandise (top items by revenue) ===")
spark.sql("""
    SELECT item_name, team, category,
           SUM(CAST(units_sold AS LONG)) AS total_units,
           ROUND(SUM(CAST(total_revenue_usd AS DOUBLE)), 2) AS total_revenue
    FROM Merchandise
    GROUP BY item_name, team, category
    ORDER BY total_revenue DESC
    LIMIT 10
""").show(truncate=False)

=== Attendance ===
+----------+--------------------+---------+----------+--------+-----------------+----------------+
|game_date |home_team           |opponent |attendance|capacity|pct_full         |gate_revenue_usd|
+----------+--------------------+---------+----------+--------+-----------------+----------------+
|2025-02-22|Colorado Rockies    |Padres   |9819      |11000   |89.30000000000000|187673.60       |
|2025-02-24|Arizona Diamondbacks|Cubs     |10237     |11000   |93.09999999999999|212420.75       |
|2025-02-26|Colorado Rockies    |Brewers  |8114      |11000   |73.80000000000000|157374.11       |
|2025-02-28|Arizona Diamondbacks|Padres   |7619      |11000   |69.30000000000000|188699.73       |
|2025-03-02|Colorado Rockies    |White Sox|10854     |11000   |98.70000000000000|254566.08       |
+----------+--------------------+---------+----------+--------+-----------------+----------------+
only showing top 5 rows
=== Concessions (top items by revenue) ===
+------------+---------

In [5]:
# Cross-sheet join: revenue per game across all streams
spark.sql("""
    SELECT
        a.game_date,
        a.home_team,
        a.opponent,
        CAST(a.attendance AS LONG)                           AS attendance,
        ROUND(CAST(a.gate_revenue_usd AS DOUBLE), 2)        AS gate_revenue,
        ROUND(SUM(CAST(c.total_revenue_usd AS DOUBLE)), 2)  AS concession_revenue,
        ROUND(SUM(CAST(m.total_revenue_usd AS DOUBLE)), 2)  AS merch_revenue
    FROM Attendance a
    LEFT JOIN Concessions c ON c.game_date = a.game_date
    LEFT JOIN Merchandise m ON m.game_date = a.game_date
    GROUP BY a.game_date, a.home_team, a.opponent, a.attendance, a.gate_revenue_usd
    ORDER BY a.game_date
""").show(20, truncate=False)

+----------+--------------------+---------+----------+------------+------------------+-------------+
|game_date |home_team           |opponent |attendance|gate_revenue|concession_revenue|merch_revenue|
+----------+--------------------+---------+----------+------------+------------------+-------------+
|2025-02-22|Colorado Rockies    |Padres   |9819      |187673.6    |547865.0          |432330.0     |
|2025-02-24|Arizona Diamondbacks|Cubs     |10237     |212420.75   |530489.5          |406905.0     |
|2025-02-26|Colorado Rockies    |Brewers  |8114      |157374.11   |603696.5          |430245.0     |
|2025-02-28|Arizona Diamondbacks|Padres   |7619      |188699.73   |603231.0          |343890.0     |
|2025-03-02|Colorado Rockies    |White Sox|10854     |254566.08   |499519.5          |480480.0     |
|2025-03-04|Arizona Diamondbacks|Mariners |9618      |213704.44   |575586.0          |447315.0     |
|2025-03-06|Colorado Rockies    |Astros   |7322      |138656.37   |541728.0          |38554